In [ ]:
import sys, pathlib
_root = next((c for c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (c / "projects" / "data").is_dir()), pathlib.Path.cwd())
sys.path.insert(0, str(_root))
from projects.data.make_synth_table import make_synth_table


# 05 · 文件 I/O：Parquet 与分块读取  IO: Parquet & Chunking

> **能力标签**：SH4（数据处理与分析）、SH5（数据工程）

## 目标 Objectives

完成本课后，你应该能够：

1. 用 `df.to_parquet()` / `pd.read_parquet()` 往返（roundtrip）Parquet 文件，并理解它保留 dtype。
2. 理解 Parquet 格式的优势（列式存储、压缩、dtype 保留）。
3. 用 `pd.read_csv(chunksize=N)` 分块读取大文件，避免一次性加载到内存。
4. 实现 `roundtrip_parquet` 与 `chunked_row_count` —— W3-io 题包核心函数。

> 对应能力：**SH4 · SH5**


## 概念讲解 Concepts

### Parquet 格式

Parquet 是 Apache Hadoop 生态的**列式存储**（columnar storage）格式：

| 特性 | CSV | Parquet |
|------|-----|---------|
| 存储方式 | 行式 | 列式 |
| dtype 保留 | 否（全为字符串）| **是** |
| 压缩 | 无 | Snappy / ZSTD |
| 读取速度 | 慢（全行扫描）| 快（按列跳过）|

```python
df.to_parquet("data.parquet")            # 写入
df2 = pd.read_parquet("data.parquet")    # 读回，dtype 与原始完全一致
```

---

### 分块读取（chunking）

对于超出内存的 CSV 文件，用 `chunksize` 参数获取 **TextFileReader 迭代器**：

```python
total = 0
for chunk in pd.read_csv("big.csv", chunksize=1000):
    total += len(chunk)   # chunk 是普通 DataFrame，行数 <= 1000
print("总行数:", total)
```

每个 `chunk` 独立处理后可立即丢弃，内存占用保持在 `chunksize` 级别。

---

### 核心函数签名（题包）

```python
def roundtrip_parquet(df, path):
    df.to_parquet(path)
    return pd.read_parquet(path)

def chunked_row_count(csv_path, chunksize=100):
    total = 0
    for chunk in pd.read_csv(csv_path, chunksize=chunksize):
        total += len(chunk)
    return total
```


## 示例 Worked Example

In [ ]:
import pandas as pd
import numpy as np
import tempfile
from pathlib import Path
from projects.data.make_synth_table import make_synth_table

# 生成示例数据
df = make_synth_table(n=200, seed=0)
print('=== 原始 dtypes ===')
print(df.dtypes)

# 1. roundtrip parquet
with tempfile.TemporaryDirectory() as tmp:
    pq_path = Path(tmp) / 'data.parquet'
    df.to_parquet(pq_path)
    df_back = pd.read_parquet(pq_path)
    print('\n=== parquet roundtrip 一致:', df.equals(df_back), '===')
    print('行数:', len(df_back), '列数:', df_back.shape[1])

# 2. CSV 分块读取
with tempfile.TemporaryDirectory() as tmp:
    csv_path = Path(tmp) / 'data.csv'
    df.to_csv(csv_path, index=False)
    total = 0
    for chunk in pd.read_csv(csv_path, chunksize=50):
        total += len(chunk)
    print('\n=== 分块读取总行数:', total, '（应为 200）===')


## 动手 Exercise

在下面的 cell 中：

1. 生成 `make_synth_table(n=500, seed=1)` 数据。
2. 实现 `roundtrip_parquet(df, path)`，写入临时文件再读回，验证 `df.equals(df_back)`。
3. 把数据存为 CSV，实现 `chunked_row_count(csv_path, chunksize=80)`，确认等于 500。


In [ ]:
import pandas as pd
import tempfile
from pathlib import Path
from projects.data.make_synth_table import make_synth_table

df = make_synth_table(n=500, seed=1)

def roundtrip_parquet(df, path):
    df.to_parquet(path)
    return pd.read_parquet(path)

def chunked_row_count(csv_path, chunksize=100):
    total = 0
    for chunk in pd.read_csv(csv_path, chunksize=chunksize):
        total += len(chunk)
    return total

# 验证
with tempfile.TemporaryDirectory() as tmp:
    pq_path = Path(tmp) / 'out.parquet'
    df_back = roundtrip_parquet(df, pq_path)
    assert df.equals(df_back), "Parquet roundtrip mismatch!"
    print('Parquet roundtrip: OK')

    csv_path = Path(tmp) / 'out.csv'
    df.to_csv(csv_path, index=False)
    count = chunked_row_count(csv_path, chunksize=80)
    assert count == 500, f"Expected 500 rows, got {count}"
    print(f'Chunked row count: {count} (expected 500)')


## 延伸阅读 Further Reading

1. **pandas 官方文档 — Parquet**: <https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html>
2. **Apache Parquet 格式规范**: <https://parquet.apache.org/docs/>
3. **pandas 官方文档 — read_csv chunksize**: <https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html>
4. **《Python for Data Analysis》第 6 章** — Wes McKinney


## 关联练习 Related Assignments

本课对应 **W3-io** 题包，核心函数为 `roundtrip_parquet` 与 `chunked_row_count`：

```bash
python check.py 03-io
```

> 能力标签：**SH4 · SH5** · threshold ≥ 0.7
